In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_feather('champions_db.feather')

In [4]:
df.head(3)

,name,image,tags,hp,hpperlevel,mp,mpperlevel,movespeed,armor,armorperlevel,spellblock,spellblockperlevel,attackrange,hpregen,hpregenperlevel,mpregen,mpregenperlevel,crit,critperlevel,attackdamage,attackdamageperlevel,attackspeed,attackspeedperlevel,image_path,image_url
0,Aatrox,Aatrox.png,"[Fighter, Tank]",650,114,0,0.0,345,38,4.45,32,2.05,175,3.0,1.0,0.0,0.0,0,0,60,5.0,0.651,2.5,images/Aatrox.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...
1,Ahri,Ahri.png,"[Mage, Assassin]",590,104,418,25.0,330,21,4.70,30,1.30,550,2.5,0.6,8.0,0.8,0,0,53,3.0,0.668,2.2,images/Ahri.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...
2,Akali,Akali.png,[Assassin],600,119,200,0.0,345,23,4.70,37,2.05,125,9.0,0.9,50.0,0.0,0,0,62,3.3,0.625,3.2,images/Akali.png,http://ddragon.leagueoflegends.com/cdn/14.8.1/...


In [5]:
# O mp do Viego vem 10_000, arrumando para 0 para não dar outlier
df.loc[df['name'] == 'Viego', 'mp'] = 0

# O mp da Bel'Veth vem 480, arrumando para 0 para não dar outlier
df.loc[df['name'] == "Bel'Veth", 'mp'] = 0

### PCA 2D

In [6]:
scaler = MinMaxScaler()

df_scaled = scaler.fit_transform(
    # df
    df[['hp', 'mp','movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']]
    .select_dtypes(include=['number']))

In [7]:
df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']][df['name'].isin(['Aatrox', 'Viego', """Bel'Veth""", 'Garen', 'Zed', 'Ziggs', 'Graves'])]

,hp,mp,movespeed,armor,spellblock,attackrange,attackspeed,attackdamage
0,650,0,345,38,32,175,0.651,60
13,610,0,340,32,32,175,0.850,60
36,690,0,340,38,32,175,0.625,69
39,625,325,340,33,32,425,0.475,68
148,630,0,345,34,32,200,0.658,57
161,654,200,345,32,29,125,0.651,63
163,606,480,325,21,30,550,0.656,55


In [8]:
pca_2d = PCA(n_components=2)
df_pca2d = pd.DataFrame(
    pca_2d.fit_transform(
        # df_scaled
        df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']]
    ),
    columns=['PC1', 'PC2']
)

In [9]:
explained_variance = pca_2d.explained_variance_
explained_variance_ratio = pca_2d.explained_variance_ratio_
print(explained_variance, explained_variance_ratio, np.cumsum(pca_2d.explained_variance_ratio_))

[41632.97595184 11221.71435889] [0.76817958 0.20705442] [0.76817958 0.975234  ]


In [10]:
df_pca2d['champion'] = df['name'].values
df_pca2d['image'] = df['image_path'].values

In [11]:
df_pca2d

,PC1,PC2,champion,image
0,-257.305906,-236.945595,Aatrox,images/Aatrox.png
1,246.613405,17.832259,Ahri,images/Ahri.png
2,-227.614724,-34.061234,Akali,images/Akali.png
3,172.251530,-26.535159,Akshan,images/Akshan.png
4,-181.639676,108.854698,Alistar,images/Alistar.png
...,...,...,...,...
162,139.112219,-120.918635,Zeri,images/Zeri.png
163,267.441811,76.253910,Ziggs,images/Ziggs.png
164,260.014171,49.078756,Zilean,images/Zilean.png
165,245.294214,25.763029,Zoe,images/Zoe.png


In [ ]:
pio.renderers.default = "vscode"

fig = go.Figure()

for _, row in df_pca2d.iterrows():
    fig.add_layout_image(
        dict(
            source=row["image"],
            x=row["PC1"],
            y=row["PC2"],
            xref="x",
            yref="y",
            sizex=25,
            sizey=25,
            xanchor="center",
            yanchor="middle",
            layer="above"
        )
    )

fig.update_xaxes(title="PC1")
fig.update_yaxes(title="PC2")

fig.update_layout(
    xaxis=dict(range=[df_pca2d["PC1"].min()-50, df_pca2d["PC1"].max()+50]),
    yaxis=dict(range=[df_pca2d["PC2"].min()-50, df_pca2d["PC2"].max()+50]),
    height=800
)

fig.show()

### PCA 3D

In [13]:
pca_3d = PCA(n_components=3)
df_pca3d = pd.DataFrame(
    pca_3d.fit_transform(
        # df_scaled
        df[['hp', 'mp', 'movespeed', 'armor', 'spellblock', 'attackrange', 'attackspeed', 'attackdamage']]
    ),
    columns=['PC1', 'PC2', 'PC3']
)

In [14]:
explained_variance = pca_3d.explained_variance_
explained_variance_ratio = pca_3d.explained_variance_ratio_
print(explained_variance, explained_variance_ratio, np.cumsum(pca_3d.explained_variance_ratio_))

[41632.97595184 11221.71435889  1271.57746689] [0.76817958 0.20705442 0.02346217] [0.76817958 0.975234   0.99869617]


In [15]:
df_pca3d['champion'] = df['name'].values
df_pca3d['image'] = df['image_path'].values

In [ ]:
pio.renderers.default = "browser"
#pio.renderers.default = "vscode"

fig = px.scatter_3d(
    df_pca3d,
    x='PC1',
    y='PC2',
    z='PC3',
    text='champion',
)

# cria lista de campeões
champions = df_pca3d['champion'].unique()

# cria botões
buttons = []

for champ in champions:
    visible = [c == champ for c in df_pca3d['champion']]

    buttons.append(
        dict(
            label=champ,
            method="update",
            args=[{
                "marker": {
                    "size": [25 if v else 3 for v in visible],
                    "opacity": [1 if v else 0.2 for v in visible]
                }
            }]
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True
        )
    ]
)

fig.show()